# 04_preprocess_rb.ipynb
## Purpose
Normalize/clean segments for the RB branch, including language fallback if used.

## Expected inputs
- `data/segments_index.csv`

## Expected outputs
- `intermediate processed text columns used by RB extraction`

## Notes
- Keys are aligned using `seg_key = comment_id__seg_id`.

In [ ]:
# =========================
# 0) Setup
# =========================
import re, json
from dataclasses import dataclass
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import pandas as pd

print("Ready.")

## 1) I/O configuration

In [ ]:
# =========================
# 1) I/O paths
# =========================
INPUT_CSV   = Path("data/dataset.csv")
TEXT_COL    = "seg_text"   # kolom segmen

SLANG_FILE  = Path("data/slangword.txt")      # opsional (slang:formal)
CUSTOM0_FILE = None                      # opsional (1 kata per baris)
CUSTOM_STOP_FILE =Path("data/custom_stopwords.txt")                   # opsional (1 kata per baris)

OUT_CSV     = Path("data/preprocessedfor_rb_with_meta.csv")
AUDIT_JSON  = Path("data/audit_preprocess_rb.json")
AUDIT_SAMPLES = Path("data/audit_samples_prep_rb.csv")

KEEP_NUMBERS = "keep"    # keep|remove  (token deprecated)

INPUT_CSV, OUT_CSV

## 2) Utility Loader

In [ ]:
def _safe_read_text(path: Path, encoding: str = "utf-8") -> str:
    return path.read_text(encoding=encoding, errors="ignore")


def load_slang_map(path: Optional[Path]) -> Dict[str, str]:
    slang = {}
    if not path:
        return slang
    if not path.exists():
        print(f"[WARN] slang file not found: {path}")
        return slang
    raw = _safe_read_text(path)
    for line in raw.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if ":" not in line:
            continue
        k, v = line.split(":", 1)
        k = k.strip().lower()
        v = v.strip().lower()
        if k and v:
            slang[k] = v
    return slang


def parse_words_file(path: Optional[Path]) -> List[str]:
    if not path:
        return []
    path = Path(path)
    if not path.exists():
        print(f"[WARN] words file not found: {path}")
        return []
    raw = _safe_read_text(path)
    out = []
    for line in raw.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        out.append(line)
    return out


def try_load_stopwords() -> Tuple[set, set]:
    try:
        import nltk  # noqa
        from nltk.corpus import stopwords
        try:
            indo = set(stopwords.words("indonesian"))
            eng  = set(stopwords.words("english"))
            return indo, eng
        except LookupError:
            print("[WARN] NLTK stopwords not downloaded. Run: nltk.download('stopwords')")
            return set(), set()
    except Exception:
        return set(), set()


slang_map = load_slang_map(SLANG_FILE if SLANG_FILE else None)
custom_words_0 = parse_words_file(CUSTOM0_FILE)
custom_stopwords = parse_words_file(CUSTOM_STOP_FILE)

indo_sw, eng_sw = try_load_stopwords()

print(f"Loaded slang entries: {len(slang_map)}")
print(f"NLTK stopwords: indo={len(indo_sw)}, eng={len(eng_sw)}")
print(f"Custom0: {len(custom_words_0)}, Custom stopwords: {len(custom_stopwords)}")

## 3) Core preprocessing + emoticon/emoji elimination

In [ ]:
# -----------------------------
# Regex precompiled
# -----------------------------
_URL_RE = re.compile(r"http\S+|www\.\S+", flags=re.IGNORECASE)
_MENTION_RE = re.compile(r"@[A-Za-z0-9_]+")
_RT_RE = re.compile(r"\bRT\b", flags=re.IGNORECASE)
_HASHTAG_RE = re.compile(r"#([A-Za-z0-9_]+)")
_MULTI_PUNCT_RE = re.compile(r"([.!?]){2,}")
_WHITESPACE_RE = re.compile(r"\s+")
_NUM_RE = re.compile(r"\b\d+(?:[\.,]\d+)?\b")

# Legacy toton that may already exist from older pipelines: <NUM>
_NUM_TOKEN_RE = re.compile(r"<\s*num\s*>", flags=re.IGNORECASE)

# Keep letters, digits, spaces, basic punctuation, and hyphen
_ALLOWED_CHARS_RE = re.compile(r"[^0-9A-Za-zÀ-ÖØ-öø-ÿ\s\.\,\!\?\-]+")

# -----------------------------
# Energy orthography canonicalization (whitelist-based, safe)
# -----------------------------
# Menyatukan variasi ejaan and frasa domain-energi yang sering muncul di diskusi publik
# (mis. batu-bara vs batu bara vs batubara) agar stabil for rule-based extraction.
#
# Prinsip:
# 1) Whitelist only (not agresif).
# 2) Dua stage:
#    (a) Regex replacement for varian based spasi/hyphen/underscore.
#    (b) Join bigram toton for frasa energi yang umum (JOIN_RULES_2).

JOIN_RULES_2 = {
    ('batu', 'bara'): 'batubara',
    ('panas', 'bumi'): 'panas bumi',
    ('gas', 'bumi'): 'gas bumi',
    ('minyak', 'bumi'): 'minyak bumi',
    ('minyak', 'mentah'): 'minyak mentah',
    ('tagihan', 'listrik'): 'tagihan listrik',
    ('tarif', 'listrik'): 'tarif listrik',
    ('harga', 'listrik'): 'harga listrik',
    ('biaya', 'listrik'): 'biaya listrik',
    ('harga', 'energi'): 'harga energi',
    ('energi', 'terbarukan'): 'energi terbarukan',
    ('bahan', 'bakar'): 'bahan bakar',
    ('carbon', 'capture'): 'carboncapture',
    ('jaringan', 'listrik'): 'jaringan listrik',
    ('kendaraan', 'listrik'): 'kendaraan listrik',
    ('mobil', 'listrik'): 'mobil listrik',
    ('token', 'listrik'): 'token listrik',
    ('byar', 'pet'): 'pemadaman',
    ('black', 'out'): 'pemadaman',
    ('kapasitas', 'daya'): 'daya',
    ('pembangkit', 'uap'): 'pltu',
    ('pembangkit', 'gas'): 'pltg',
    ('mini', 'hidro'): 'pltm',
    ('mini', 'hydro'): 'pltm',
    ('makro', 'hidro'): 'pltm',
    ('macro', 'hydro'): 'pltm',
    ('transisi', 'energi'): 'transisi energi',
    ('green', 'energy'): 'energi hijau',
    ('ramah', 'lingkungan'): 'ramah lingkungan',
    ('net', 'zero'): 'net zero',
    ('polusi', 'udara'): 'polusi udara',
    ('base', 'load'): 'baseload',
    ('reserve', 'margin'): 'reserve margin',
    ('cadangan', 'daya'): 'cadangan daya',
    ('kapasitas', 'cadangan'): 'kapasitas cadangan',
}

_ENERGY_VARIANTS = [
    ('bahan bakar', '\\bbahan[\\s\\-_]*bakar\\b'),
    ('base load', '\\bbase[\\s\\-_]*load\\b'),
    ('batubara', '\\bbatu[\\s\\-_]*bara\\b'),
    ('biaya listrik', '\\bbiaya[\\s\\-_]*listrik\\b'),
    ('cadangan daya', '\\bcadangan[\\s\\-_]*daya\\b'),
    ('carboncapture', '\\bcarbon[\\s\\-_]*capture\\b'),
    ('daya', '\\bkapasitas[\\s\\-_]*daya\\b'),
    ('energi hijau', '\\bgreen[\\s\\-_]*energy\\b'),
    ('energi terbarukan', '\\benergi[\\s\\-_]*terbarukan\\b'),
    ('gas bumi', '\\bgas[\\s\\-_]*bumi\\b'),
    ('harga energi', '\\bharga[\\s\\-_]*energi\\b'),
    ('harga listrik', '\\bharga[\\s\\-_]*listrik\\b'),
    ('jaringan listrik', '\\bjaringan[\\s\\-_]*listrik\\b'),
    ('kapasitas cadangan', '\\bkapasitas[\\s\\-_]*cadangan\\b'),
    ('kendaraan listrik', '\\bkendaraan[\\s\\-_]*listrik\\b'),
    ('minyak bumi', '\\bminyak[\\s\\-_]*bumi\\b'),
    ('minyak mentah', '\\bminyak[\\s\\-_]*mentah\\b'),
    ('mobil listrik', '\\bmobil[\\s\\-_]*listrik\\b'),
    ('net zero', '\\bnet[\\s\\-_]*zero\\b'),
    ('panas bumi', '\\bpanas[\\s\\-_]*bumi\\b'),
    ('pemadaman', '\\bblack[\\s\\-_]*out\\b'),
    ('pemadaman', '\\bbyar[\\s\\-_]*pet\\b'),
    ('pltg', '\\bpembangkit[\\s\\-_]*gas\\b'),
    ('pltm', '\\bmakro[\\s\\-_]*hidro\\b'),
    ('pltm', '\\bmacro[\\s\\-_]*hydro\\b'),
    ('pltm', '\\bmini[\\s\\-_]*hidro\\b'),
    ('pltm', '\\bmini[\\s\\-_]*hydro\\b'),
    ('pltu', '\\bpembangkit[\\s\\-_]*uap\\b'),
    ('polusi udara', '\\bpolusi[\\s\\-_]*udara\\b'),
    ('ramah lingkungan', '\\bramah[\\s\\-_]*lingkungan\\b'),
    ('reserve margin', '\\breserve[\\s\\-_]*margin\\b'),
    ('tagihan listrik', '\\btagihan[\\s\\-_]*listrik\\b'),
    ('tarif listrik', '\\btarif[\\s\\-_]*listrik\\b'),
    ('token listrik', '\\btoken[\\s\\-_]*listrik\\b'),
    ('transisi energi', '\\btransisi[\\s\\-_]*energi\\b'),
]

# compile sekali
_ENERGY_VARIANT_RES = [(canon, re.compile(pat)) for canon, pat in _ENERGY_VARIANTS]

def normalize_energy_variants(t: str) -> str:
    """Regex-based whitelist replacement (spasi/hyphen/underscore)."""
    if not t:
        return t
    for canon, rx in _ENERGY_VARIANT_RES:
        t = rx.sub(canon, t)
    return t

def apply_join_rules_text(t: str) -> str:
    """Join bigram whitelist pada level token (mis. 'tagihan listrik' -> 'tagihan listrik')."""
    if not t:
        return t
    toks = t.split()
    out = []
    i = 0
    while i < len(toks):
        if i < len(toks) - 1:
            key = (toks[i], toks[i+1])
            rep = JOIN_RULES_2.get(key)
            if rep is not None:
                out.append(rep)
                i += 2
                continue
        out.append(toks[i])
        i += 1
    return " ".join(out)

# -----------------------------
# Emoticon / emoji removal
# -----------------------------
# Kaskus-style: :ngakak  or :ngakak:
KASKUS_EMO_RE = re.compile(
    r"(?<!\w):[A-Za-z][\w\-]{1,30}:(?!\w)|(?<!\w):[A-Za-z][\w\-]{1,30}(?!\w)"
)

# ASCII emoticons: :) ;-) :D :P etc.
ASCII_EMO_RE = re.compile(r"(?::|;|=)(?:-)?(?:\)|\(|D|P|p|/|\\)")

# Emoji unicode blocks
EMOJI_RE = re.compile(
    "[" 
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA70-\U0001FAFF"
    "\u2600-\u26FF"
    "\u2700-\u27BF"
    "]+"
)

# ZWJ & variation selector
ZWJ_VS_RE = re.compile(r"[\u200d\uFE0F]")


@dataclass
class PreprocessConfig:
    keep_numbers: str = "keep"  # keep|remove (token deprecated)
    keep_hashtag_words: bool = True
    preserve_punct: bool = True
    lowercase: bool = True


def normalize_slang(text: str, slang_map: Dict[str, str]) -> str:
    if not text or not slang_map:
        return text
    t = text

    multi = [(k, v) for k, v in slang_map.items() if " " in k]
    multi.sort(key=lambda kv: len(kv[0]), reverse=True)
    for k, v in multi:
        pattern = re.compile(r"(?<!\w)" + re.escape(k) + r"(?!\w)")
        t = pattern.sub(v, t)

    for k, v in ((k, v) for k, v in slang_map.items() if " " not in k):
        pattern = re.compile(r"\b" + re.escape(k) + r"\b")
        t = pattern.sub(v, t)

    return t


def clean_text(text: str, cfg: PreprocessConfig) -> str:
    if text is None:
        return ""
    t = str(text)

    if cfg.lowercase:
        t = t.lower()

    t = _URL_RE.sub(" ", t)
    t = _MENTION_RE.sub(" ", t)
    t = _RT_RE.sub(" ", t)

    if cfg.keep_hashtag_words:
        t = _HASHTAG_RE.sub(r"\1", t)
    else:
        t = _HASHTAG_RE.sub(" ", t)

    t = t.replace("\n", " ").replace("\r", " ")

    # If upstream pipeline already replaced digits with <NUM>, prevent it from becoming "num"
    # by removing the toton early. (We cannot recover the original digits from <NUM>.)
    t = _NUM_TOKEN_RE.sub(" ", t)

    # IMPORTANT: remove emoticons/emoji EARLY (before whitelist)
    t = KASKUS_EMO_RE.sub(" ", t)
    t = ASCII_EMO_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = ZWJ_VS_RE.sub(" ", t)

# ====
    # reduplikasi ala forum: negara2 -> negara-negara
    # HATI-HATI: jangan change CO2, H2O, G20, dll
    EXC_REdup2 = {"co2", "h2o", "g20"}
    
    REDUP2_RE = re.compile(r"\b([a-z]{3,})2\b", flags=re.IGNORECASE)
    
    def _redup2(m):
        w = m.group(1)
        if (w + "2").lower() in EXC_REdup2:
            return w + "2"
        # return f"{w}-{w}"
        return f"{w}"
    
    t = REDUP2_RE.sub(_redup2, t)
# ====

    # Numbers:
    # - default is KEEP digits as-is (do not map to a special toton)
    if cfg.keep_numbers == "remove":
        t = _NUM_RE.sub(" ", t)
    elif cfg.keep_numbers in ("keep", "token"):
        # 'toton' topt for backward compatibility; treated as 'toep'
        pass
    else:
        raise ValueError("keep_numbers must be one of: keep|remove")


    if cfg.preserve_punct:
        t = _MULTI_PUNCT_RE.sub(r"\1", t)
    else:
        t = re.sub(r"[.!?,]", " ", t)

    t = _ALLOWED_CHARS_RE.sub(" ", t)
    t = _WHITESPACE_RE.sub(" ", t).strip()
    t = normalize_energy_variants(t)
    t = apply_join_rules_text(t)
    t = _WHITESPACE_RE.sub(" ", t).strip()
    return t


def sentence_capitalize(text: str) -> str:
    if not text:
        return text
    t = text.strip()
    if t:
        t = t[0].upper() + t[1:]
    def _cap(m):
        return m.group(1) + " " + m.group(2).upper()
    t = re.sub(r"([.!?])\s+([a-z])", _cap, t)
    return t

## 4) Pipeline DataFrame for `seg_text` (parsing-safe + lexical)

In [ ]:
PUNCT_STRIP = ".,!?;:()[]{}\"'`"

def remove_stopwords(text: str, stopset: set) -> str:
    if not text or not stopset:
        return text
    out = []
    for tok in text.split():
        core = tok.strip(PUNCT_STRIP).lower()
        if not core:
            continue
        if core in stopset:
            continue
        out.append(tok)
    return " ".join(out)


def preprocess_dataframe(
    df: pd.DataFrame,
    text_col: str,
    slang_map: Dict[str, str],
    custom_words_0: Optional[Iterable[str]] = None,
    custom_stopwords: Optional[Iterable[str]] = None,
    cfg: Optional[PreprocessConfig] = None
) -> pd.DataFrame:
    cfg = cfg or PreprocessConfig()

    indo_sw, eng_sw = try_load_stopwords()
    custom_stop = set([w.strip().lower() for w in (custom_stopwords or []) if str(w).strip()])
    stopset_all = indo_sw.union(eng_sw).union(custom_stop)

    custom0 = set([w.strip().lower() for w in (custom_words_0 or []) if str(w).strip()])
    stopset_0 = eng_sw.union(custom0)

    # Keep toy metadata columns from dataset.csv for traceability
    meta_cols = [
        'comment_id','comment_ori','seg_id','source','user','date',
        text_col
    ]
    cols_existing = [c for c in meta_cols if c in df.columns]
    out = df[cols_existing].copy() if cols_existing else pd.DataFrame()

    # Always toep a raw copy used for preprocessing
    out['seg_text_raw'] = df[text_col].fillna('').astype(str)

    tmp = out["seg_text_raw"].apply(lambda x: clean_text(x, cfg))
    tmp = tmp.apply(lambda x: normalize_slang(x, slang_map))

    tmp0 = tmp.apply(lambda x: remove_stopwords(x, stopset_0)) if stopset_0 else tmp

    out["seg_text_parsing"] = tmp0
    out["seg_text_processed"] = tmp0.apply(sentence_capitalize)

    out["filtered1_lexical"] = out["seg_text_processed"].apply(lambda x: remove_stopwords(x, stopset_all))

    return out

## 5) Audit (including emoticon/emoji before-after)

In [ ]:
def tokenize_simple(text: str) -> List[str]:
    if not text:
        return []
    return re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9]+", str(text).lower())


def has_kaskus_emo(text: str) -> bool:
    if text is None:
        return False
    return bool(KASKUS_EMO_RE.search(str(text)))


def has_emoji(text: str) -> bool:
    if text is None:
        return False
    return bool(EMOJI_RE.search(str(text)))


def build_audit(df_out: pd.DataFrame, sample_n: int = 60) -> Tuple[dict, pd.DataFrame]:
    raw_col = "seg_text_raw"
    proc_col = "seg_text_processed"

    raw_tokens = Counter()
    proc_tokens = Counter()

    empty_raw = int((df_out[raw_col].fillna("").astype(str).str.strip() == "").sum())
    empty_proc = int((df_out[proc_col].fillna("").astype(str).str.strip() == "").sum())
    changed_rows = int((df_out[raw_col].astype(str) != df_out[proc_col].astype(str)).sum())

    # --- Energy orthography audit (whitelist variants) ---
    # Mengukur dampak normalisasi such as: batu-bara/batu bara -> batubara, dll.
    variant_defs = [
        ("batubara", r"batu[\s\-_]*bara", r"batubara"),
        ("panas bumi", r"panas[\s\-_]*bumi", r"panas\s+bumi"),
        ("gas bumi", r"gas[\s\-_]*bumi", r"gas\s+bumi"),
        ("minyak bumi", r"minyak[\s\-_]*bumi", r"minyak\s+bumi"),
        ("energi terbarukan", r"energi[\s\-_]*terbarukan", r"energi\s+terbarukan"),
        ("carbon capture", r"carbon[\s\-_]*capture", r"carbon\s+capture"),
    ]
    energy_variants = {}
    raw_series = df_out[raw_col].fillna("").astype(str).str.lower()
    proc_series = df_out[proc_col].fillna("").astype(str).str.lower()

    for canon, pat_before, pat_after in variant_defs:
        rx_before = re.compile(pat_before)
        rx_after  = re.compile(pat_after)
        n_before = int(raw_series.str.contains(rx_before, regex=True).sum())
        n_after  = int(proc_series.str.contains(rx_after, regex=True).sum())
        n_fixed  = int((raw_series.str.contains(rx_before, regex=True) & proc_series.str.contains(rx_after, regex=True)).sum())
        energy_variants[canon] = {"hits_before": n_before, "hits_after": n_after, "rows_fixed": n_fixed}


    kaskus_before = int(df_out[raw_col].apply(has_kaskus_emo).sum())
    kaskus_after  = int(df_out[proc_col].apply(has_kaskus_emo).sum())
    emoji_before  = int(df_out[raw_col].apply(has_emoji).sum())
    emoji_after   = int(df_out[proc_col].apply(has_emoji).sum())

    for r, p in zip(df_out[raw_col].fillna(""), df_out[proc_col].fillna("")):
        raw_tokens.update(tokenize_simple(r))
        proc_tokens.update(tokenize_simple(p))

    removed = raw_tokens.copy()
    for k, v in proc_tokens.items():
        removed[k] -= v
        if removed[k] <= 0:
            del removed[k]

    audit = {
        "rows": int(len(df_out)),
        "empty_raw": empty_raw,
        "empty_processed": empty_proc,
        "changed_rows": changed_rows,
        "energy_variants": energy_variants,
        "vocab_raw": int(len(raw_tokens)),
        "vocab_processed": int(len(proc_tokens)),
        "top_tokens_raw": raw_tokens.most_common(30),
        "top_tokens_processed": proc_tokens.most_common(30),
        "top_tokens_removed": removed.most_common(30),
        "emoticon_emoji": {
            "has_kaskus_emo_before": kaskus_before,
            "has_kaskus_emo_after": kaskus_after,
            "has_emoji_before": emoji_before,
            "has_emoji_after": emoji_after,
        }
    }

    samples = df_out[[raw_col, proc_col]].copy()
    samples["has_kaskus_emo_before"] = df_out[raw_col].apply(has_kaskus_emo)
    samples["has_kaskus_emo_after"]  = df_out[proc_col].apply(has_kaskus_emo)
    samples["has_emoji_before"]      = df_out[raw_col].apply(has_emoji)
    samples["has_emoji_after"]       = df_out[proc_col].apply(has_emoji)
    samples["changed"] = samples[raw_col].astype(str) != samples[proc_col].astype(str)

    priority = samples[(samples["has_kaskus_emo_before"]) | (samples["has_emoji_before"]) | (samples["changed"])].head(sample_n)
    if len(priority) < sample_n:
        priority = pd.concat([priority, samples.head(sample_n - len(priority))], ignore_index=True)

    return audit, priority

## 6) Run & Save

In [ ]:
# =========================
# 6) Run
# =========================
assert INPUT_CSV.exists(), f"Input file not found: {INPUT_CSV}"

df = pd.read_csv(INPUT_CSV)
assert TEXT_COL in df.columns, f"Column '{TEXT_COL}' not found. Available: {list(df.columns)[:30]}"

cfg = PreprocessConfig(
    keep_numbers=KEEP_NUMBERS,
    keep_hashtag_words=True,
    preserve_punct=True,
    lowercase=True,
)

df_out = preprocess_dataframe(
    df=df,
    text_col=TEXT_COL,
    slang_map=slang_map,
    custom_words_0=custom_words_0,
    custom_stopwords=custom_stopwords,
    cfg=cfg
)

df_out.to_csv(OUT_CSV, index=False, encoding="utf-8")
print("[OK] Saved:", OUT_CSV, "| rows:", len(df_out))

audit, samples = build_audit(df_out, sample_n=60)
AUDIT_JSON.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
samples.to_csv(AUDIT_SAMPLES, index=False, encoding="utf-8")

print("[OK] Saved:", AUDIT_JSON)
print("[OK] Saved:", AUDIT_SAMPLES)

samples.head(12)

## 7) Quick test

In [ ]:
tests = [
    "Inget gan, masih ada kaum kapitalis yang nggak mau rugi :toast",
    "Heheheeeeee ane kebetulan pengamat gan 😂😂😂",
    "nice share gan :thumbup",
    "indoensia sebenarnya kaya bangett :matabelo: mewek",
    "capek :capedes tapi lanjut :)",
    "wkwk :ngakak: ini lucu 🤣🤣",
]
for s in tests:
    print("BEFORE:", s)
    print("AFTER :", clean_text(s, cfg))
    print("-"*70)